# Co-occurrence: which core groups occur together, and their onset

The same questions for the five core groups the analyses use (and LGS): how often two groups occur in the same patient, and how each group's patients split by onset (generalized only, focal only, or both). Patients: everyone with pre-op types recorded.

The codes counterpart is `4_combinations/codes/cooccurrence.ipynb`. Saved to `results/<version>/4_combinations/types/figures/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cc_stats.build_analysis import CORE_TYPES
from cc_stats.paths import DATASET
from cc_stats.plots import GROUP_COLORS, pretty, save_figure

print(f'dataset {DATASET}')

In [ ]:
from cc_stats.paths import ANALYSIS_CSV

analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
has_cols = [c for c in analysis.columns if c.startswith('has_')]
patients = analysis.dropna(subset=has_cols)
FEATURES = CORE_TYPES + ['lgs']
present = pd.DataFrame({t: patients['lgs'] if t == 'lgs' else patients[f'has_{t}'] for t in FEATURES})
phi = present.corr()
both = present.fillna(0).T @ present.fillna(0)

# Lower triangle, without the empty first row and last column
rows, cols = FEATURES[1:], FEATURES[:-1]
fig, ax = plt.subplots(figsize=(7, 5.5))
grid = phi.loc[rows, cols].where(np.tril(np.ones((len(rows), len(cols)), dtype=bool)))
im = ax.imshow(grid.to_numpy(dtype=float), cmap='RdBu_r', vmin=-1, vmax=1)
for i, r in enumerate(rows):
    for j, c in enumerate(cols):
        if j <= i:
            ax.text(j, i, f'{phi.loc[r, c]:.2f}\n(n={int(both.loc[r, c])})', ha='center', va='center', fontsize=9)
names = {t: pretty(t) for t in FEATURES}
ax.set_xticks(range(len(cols)), [names[c] for c in cols])
ax.set_yticks(range(len(rows)), [names[r] for r in rows])
for ticks, keys in ((ax.get_xticklabels(), cols), (ax.get_yticklabels(), rows)):
    for tick, k in zip(ticks, keys):
        tick.set_color(GROUP_COLORS.get(k, 'black'))
        tick.set_fontweight('bold' if k == 'lgs' else 'normal')
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, label='phi (n = patients with both)')
ax.set_title(f'Which core groups occur together (n={len(patients)})', loc='left')
plt.tight_layout()
save_figure('4_combinations/types', 'core_group_cooccurrence')
plt.show()

In [ ]:
# Onset of each core group's patients: generalized only, both, focal only (and
# the few with neither, e.g. only unknown-onset types besides the core group)
onset = pd.DataFrame({t: {
    'generalized only': int((patients[f'has_{t}'].eq(1) & patients['has_pure_general'].eq(1)).sum()),
    'both': int((patients[f'has_{t}'].eq(1) & patients['has_both_focal_general'].eq(1)).sum()),
    'focal only': int((patients[f'has_{t}'].eq(1) & patients['has_pure_focal'].eq(1)).sum()),
    'n': int(patients[f'has_{t}'].eq(1).sum())} for t in CORE_TYPES}).T
onset['neither'] = onset['n'] - onset[['generalized only', 'both', 'focal only']].sum(axis=1)
display(onset)

categories = [c for c in ('generalized only', 'both', 'focal only', 'neither') if onset[c].sum() > 0]
share = onset[categories].div(onset['n'], axis=0)
fig, ax = plt.subplots(figsize=(8, 3.6))
left = np.zeros(len(share))
for col, color in zip(share.columns, ['#3b6fb6', '#8fa8c8', '#d1872c', '#cccccc']):
    ax.barh(share.index, share[col], left=left, color=color, label=col)
    for y, (l, v) in enumerate(zip(left, share[col])):
        if v >= 0.06:
            ax.text(l + v / 2, y, f'{v:.0%}', ha='center', va='center', fontsize=9)
    left += share[col].to_numpy()
ax.set_yticks(range(len(share)), [f'{pretty(t)} (n={onset.loc[t, "n"]})' for t in share.index])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title('Onset of each core group\'s patients', loc='left')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=len(categories), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_figure('4_combinations/types', 'core_group_onset')
plt.show()